## Pre-Processing

##### The prediction task is to predict the price of a house (column price) given the other features. Please ignore the columns `id` and `date`, as well as the categorical column `zipcode`. File `kc_house_data.csv` includes all the records in the dataset. The training file `train.csv` and testing file `test.csv` include each 1000 records extracted from the dataset. Please apply the following transformations to the data before using it for this homework:

- Scale the data so that each feature has mean 0 and standard deviation 1.
- Divide the price by 1000 for all rows in the dataset. This will reduce the value of MSE.

In [80]:
import pandas as pd
import matplotlib.pyplot as plt  
from sklearn.preprocessing import StandardScaler

house_data_df = pd.read_csv('kc_house_data.csv')
all_y = house_data_df['price'] / 1000     # Reduce MSE by dividing price by 1000
house_data_df.drop(columns=['id', 'date', 'zipcode', 'price'], inplace=True)

training_df = pd.read_csv('train.csv', index_col=0) # Skip unnamed first column
training_y = training_df['price'] / 1000     # Reduce MSE by dividing price by 1000
training_df.drop(columns=['zipcode', 'price'], inplace=True)

test_df = pd.read_csv('test.csv', index_col=0) # Skip unnamed first column
test_y = test_df['price'] / 1000     # Reduce MSE by dividing price by 1000
test_df.drop(columns=['id', 'date', 'zipcode', 'price'], inplace=True)

scaler = StandardScaler()
scaler.fit(training_df)

X_train_scaled = scaler.transform(training_df)
X_test_scaled = scaler.transform(test_df)

training_df = pd.DataFrame(X_train_scaled, columns=training_df.columns)
test_df = pd.DataFrame(X_test_scaled, columns=test_df.columns)

print(training_df.head())


   bedrooms  bathrooms  sqft_living  sqft_lot    floors  waterfront      view  \
0 -0.409823  -1.449888    -0.981646 -0.312717 -0.863477   -0.089803 -0.309908   
1 -0.409823   0.283184     0.584578 -0.257719  1.070402   -0.089803 -0.309908   
2 -1.584103  -1.449888    -1.443625 -0.162440 -0.863477   -0.089803 -0.309908   
3  0.764456   1.323028    -0.102758 -0.335172 -0.863477   -0.089803 -0.309908   
4 -0.409823  -0.063430    -0.418256 -0.228769 -0.863477   -0.089803 -0.309908   

   condition     grade  sqft_above  sqft_basement  yr_built  yr_renovated  \
0  -0.673452 -0.522576   -0.722231      -0.667586 -0.498602     -0.206760   
1  -0.673452 -0.522576    0.531438       0.219976 -0.640563      4.828896   
2  -0.673452 -1.384913   -1.241427      -0.667586 -1.279387     -0.206760   
3   2.229358 -0.522576   -0.886854       1.351617 -0.143700     -0.206760   
4  -0.673452  0.339761   -0.089065      -0.667586  0.637085     -0.206760   

        lat      long  sqft_living15  sqft_lot15  

## Problem 2: Linear Regression

#### 2.1: Use an existing package to train a multiple linear regression model on the training set using all the features (except the ones excluded above). Report the coefficients of the linear regression models and the following metrics on the training data: (1) MSE metric; (2) $R^2$ metric.

In [81]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(training_df, training_y)

training_mse = mean_squared_error(training_y, model.predict(training_df))
training_r2 = model.score(training_df, training_y)

print('Intercept:', model.intercept_)
print('Coefficients:', model.coef_)
print('Training MSE:', training_mse)
print('Training R^2:', training_r2)

Intercept: 520.414834000001
Coefficients: [-12.52196187  18.52763251  56.7488368   10.88186845   8.04372084
  63.74289956  48.20010852  12.96426936  92.23147482  48.29008886
  27.13703247 -67.64311741  17.27137953  78.37573693  -1.03520308
  45.57765781 -12.93009098]
Training MSE: 31486.167775794882
Training R^2: 0.7265334318706018


#### 2.2: Evaluate the model on the testing set. Report the MSE and $R^2$ metrics on the testing set.

In [82]:
testing_mse = mean_squared_error(test_y, model.predict(test_df))
testing_r2 = model.score(test_df, test_y)

print('Testing MSE:', testing_mse)
print('Testing R^2:', testing_r2)

Testing MSE: 57628.154705670386
Testing R^2: 0.6543560876120954


#### 2.3 Interpret the results in your own words. Which features contribute mostly to the linear regression model? Is the model fitting the data well? How large is the model error? How do the training and testing MSE relate?

##### There is still a large amount of error with both training and testing data sets, as the MSE is still high (but this is more expected as there are many features). The MSE for the training set is lower than that for the testing set, which is expected as the model was optimized for the training set (but is still not great). The R^2 values for both data sets, however, are pretty high (above 0.6), which indicates the model is fitting pretty well. The features with higher values and higher standard deviations will have a higher impact on the model, like sqft_living. 

## Problem 3:  Implementing closed-form solution for linear regression

#### In this problem, you will implement your own linear regression model, using the closed-form solution we derived in class. You will also compare your model with the one trained with the package in Problem 2 on the same house price prediction dataset.

- Implement the closed-from solution for multiple linear regression using matrix operations and train a model on the training set. Write a function to predict the response on a new testing point.
- Compare the models given by your implementation with those trained in Problem 2 by the Python packages. Report the MSE and $R^2$ metrics for the models you implemented on both training and testing sets and compare these metrics to the ones given by the package implementation from Problem 2. Discuss if the results of your implementation are similar to those of the package.

In [89]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

def add_bias(x):
    return np.column_stack([np.ones(len(x)), x])

# The closed-form solution for linear regression: theta = (X^T X)^(-1) X^T y
def find_closed_form_theta(X, y):
    X_b = add_bias(X)
    theta, *_ = np.linalg.lstsq(X_b, y, rcond=None)
    return theta

def predict(X, theta):
    return X @ theta

def closed_form_solution(X, y, theta):

    # Convert dataframes to matrices and add the intercept column of ones
    X_mat = add_bias(X.values)
    y_mat = y.values

    # Make predictions
    return y_mat, predict(X_mat, theta)

theta = find_closed_form_theta(training_df.values, training_y.values)

y_train, y_training_pred = closed_form_solution(training_df, training_y, theta)
y_test, y_testing_pred = closed_form_solution(test_df, test_y, theta)

# Compute metrics
training_mse_closed = mean_squared_error(y_train, y_training_pred)
testing_mse_closed = mean_squared_error(y_test, y_testing_pred)
training_r2_closed = r2_score(y_train, y_training_pred)
testing_r2_closed = r2_score(y_test, y_testing_pred)

print("Closed-form Solution Results:")
print(f"Training MSE: {training_mse_closed}")
print(f"Testing MSE: {testing_mse_closed}")
print(f"Training R^2: {training_r2_closed}")
print(f"Testing R^2: {testing_r2_closed}")

print("\nComparison with sklearn.linear_model:")
print(f"Training MSE - Closed-form: {training_mse_closed}, sklearn.linear_model: {training_mse}")
print(f"Testing MSE - Closed-form: {testing_mse_closed}, sklearn.linear_model: {testing_mse}")
print(f"Training R^2 - Closed-form: {training_r2_closed}, sklearn.linear_model: {training_r2}")
print(f"Testing R^2 - Closed-form: {testing_r2_closed}, sklearn.linear_model: {testing_r2}")


print("\nThe results are the same (with some small floating-point precision differences), which shows that the closed-form solution \nand sklearn's LinearRegression are consistent in their predictions and performance metrics.")

Closed-form Solution Results:
Training MSE: 31486.167775794886
Testing MSE: 57628.15470567041
Training R^2: 0.7265334318706018
Testing R^2: 0.6543560876120953

Comparison with sklearn.linear_model:
Training MSE - Closed-form: 31486.167775794886, sklearn.linear_model: 31486.167775794882
Testing MSE - Closed-form: 57628.15470567041, sklearn.linear_model: 57628.154705670386
Training R^2 - Closed-form: 0.7265334318706018, sklearn.linear_model: 0.7265334318706018
Testing R^2 - Closed-form: 0.6543560876120953, sklearn.linear_model: 0.6543560876120954

The results are the same (with some small floating-point precision differences), which shows that the closed-form solution 
and sklearn's LinearRegression are consistent in their predictions and performance metrics.


## Problem 4: Polynomial Regression

#### - Consider a feature $X$, a response variable $Y$, and $N$ samples of training data. Implement a polynomial regression model that fits a polynomial of degree $p$ to the data using the least-square method. Use your own implementation from Problem 3 and adapt it for polynomial regression. If $p=2$, the model will use two features ($X$ and $X^2$), if $p=3$ the model will use 3 features ($X,X^2,X^3$), and so on for larger values of $p$.

#### - Consider the house price prediction problem with feature $X=$ sqft_living. Train a polynomial regression model for different values of $p \le 5$ using your implementation. Include a table with the MSE and $R^2$ metrics on both the training and testing data for at least 3 different values of $p$. Discuss your observations on how the MSE and $R^2$ metrics change with the degree of the polynomial.

In [91]:
def poly_features(series, degree):
    x = np.asarray(series).reshape(-1)
    return np.column_stack([x ** p for p in range(1, degree + 1)])


# Use the single feature `sqft_living` for polynomial regression
feature_name = 'sqft_living'
X_train_feat = training_df[feature_name].values
X_test_feat = test_df[feature_name].values
y_train_vals = training_y.values
y_test_vals = test_y.values

table = []

for degree in range(1, 6):
    X_train_poly = poly_features(X_train_feat, degree)
    X_test_poly = poly_features(X_test_feat, degree)

    theta_poly = find_closed_form_theta(X_train_poly, y_train_vals)

    y_train_pred = predict(add_bias(X_train_poly), theta_poly)
    y_test_pred = predict(add_bias(X_test_poly), theta_poly)

    train_mse = mean_squared_error(y_train_vals, y_train_pred)
    test_mse = mean_squared_error(y_test_vals, y_test_pred)
    train_r2 = r2_score(y_train_vals, y_train_pred)
    test_r2 = r2_score(y_test_vals, y_test_pred)

    table.append({"Degree": degree, "Train MSE": train_mse, "Test MSE": test_mse, "Train R^2": train_r2, "Test R^2": test_r2})

results_pr = pd.DataFrame(table).sort_values(["Degree", "Train MSE", "Test MSE", "Train R^2", "Test R^2"]).reset_index(drop=True)
results_pr


,Degree,Train MSE,Test MSE,Train R^2,Test R^2
0,1,57947.526161,88575.978543,0.496709,0.468736
1,2,54822.665116,71791.679479,0.523849,0.569406
2,3,53785.194716,99833.483763,0.532860,0.401216
3,4,52795.774758,250979.274285,0.541453,-0.505331
4,5,52626.111955,570616.914821,0.542927,-2.422464


#### Observations: A higher polynomial degree improves the training fit (training R^2 increases from 0.49 at p=1 to 0.54 at p=5, and MSE continually shrinks), but the testing fit improves until p=2 (at p=2, test R^2 is 0.56, and MSE decreases 88575 -> 71791) and then begins to deteriorate (at p=3, test R^2 is 0.4 and MSE is 99833, and then R^2 = -0.5, MSE = 250979 at p=4 and r^2 = -2.4, MSE = 570616 for p=5). This tells us the model is overfitting, because the higher polynomial degree overfits to the training data and poorly generalizes for the testing data.

## Problem 5:  Gradient descent (20 points)

##### In this problem, you will implement your own gradient descent algorithm and apply it to linear regression on the same house prediction dataset.

#### 1. Write code for gradient descent for training linear regression using the algorithm from class.


In [85]:
def gradient_descent_linear_regression(X, y, learning_rate=0.01, iterations=100):
    m = len(y) # Number of training examples
    
    theta = np.zeros(X.shape[1]) 

    for i in range(iterations):

        predictions = X.dot(theta)
        
        error = predictions - y
        
        gradient = (2/m) * X.T.dot(error)
        
        # Update theta
        theta = theta - learning_rate * gradient


    return theta


#### 2. Vary the value of the learning rate (at least 3 different values $\alpha \in \{0.01,0.1,0.5\}$) and report the value of the model parameter $\theta$ after different number of iterations (10, 50, and 100). Include in a table the MSE and $R^2$ metrics on the training and testing set for the different number of iterations and different learning rates. You can choose more values of the learning rates to observe how the behavior of the algorithm changes.


In [86]:
X_train_full = add_bias(training_df.values)
X_test_full = add_bias(test_df.values)
y_train_vals = training_y.values
y_test_vals = test_y.values

table = []

for learning_rate in [0.01, 0.1, 0.5]:
    for iterations in [10, 50, 100]:
        theta = gradient_descent_linear_regression(X_train_full, y_train_vals, learning_rate=learning_rate, iterations=iterations)

        y_train_pred = X_train_full @ theta
        y_test_pred = X_test_full @ theta

        train_mse = mean_squared_error(y_train_vals, y_train_pred)
        test_mse = mean_squared_error(y_test_vals, y_test_pred)
        train_r2 = r2_score(y_train_vals, y_train_pred)
        test_r2 = r2_score(y_test_vals, y_test_pred)

        table.append({"LR": learning_rate, "Iter": iterations, "theta": theta[0], "Train MSE": train_mse, "Test MSE": test_mse, "Train R^2": train_r2, "Test R^2": test_r2})

results_gd = pd.DataFrame(table).sort_values(["LR", "Iter"]).reset_index(drop=True)
results_gd

,LR,Iter,theta,Train MSE,Test MSE,Train R^2,Test R^2
0,0.01,10,9.519802e+01,2.357278e+05,2.805687e+05,-1.047365e+00,-6.828036e-01
1,0.01,50,3.308955e+02,6.972050e+04,9.704954e+04,3.944571e-01,4.179133e-01
2,0.01,100,4.513976e+02,3.682035e+04,6.333304e+04,6.802045e-01,6.201392e-01
3,0.10,10,4.645357e+02,3.510510e+04,6.163043e+04,6.951019e-01,6.303511e-01
4,0.10,50,5.204074e+02,3.149726e+04,5.772248e+04,7.264371e-01,6.537904e-01
5,0.10,100,5.204148e+02,3.148643e+04,5.763896e+04,7.265311e-01,6.542913e-01
6,0.50,10,5.204148e+02,1.456064e+17,1.626068e+17,-1.264635e+12,-9.752880e+11
7,0.50,50,2.245447e+18,1.259542e+67,1.406601e+67,-1.093949e+62,-8.436553e+61
8,0.50,100,3.687170e+49,3.322792e+129,3.710745e+129,-2.885942e+124,-2.225642e+124


#### 3. Write some observations about the behavior of the algorithm: How do the metrics change with different learning rates; How many iterations are needed; Does the algorithm converge to the optimal solution, etc.

##### The MSE and R^2 for training and test sets both initially improve with more iterations and a higher learning rate, at first with most drastic improvements (between 10 and 50 iterations of 0.01 LR, the MSE is divided by roughly 3 and the R^2 goes from negative to positive), then slows (between 50 and 100 iterations at 0.1 LR, the MSE and R^2 are almost the same) until the point of 0.5 LR and 10 iterations, at which point the MSE and R^2 skyrocket, signaling divergence.

## Problem 6: Ridge regularization

#### 2. Modify your implementation from Problem 5 to implement ridge regression with gradient descent.

In [87]:
def gradient_descent_ridge_regression(X, y, learning_rate=0.01, iterations=100, lambda_reg=0.1):
    m = len(y)
    theta = np.zeros(X.shape[1])

    for i in range(iterations):
        predictions = X.dot(theta)
        error = predictions - y
        
        # Gradient with L2 regularization
        gradient = (2/m) * X.T.dot(error) + (2 * lambda_reg / m) * theta
        
        # Update theta
        theta = theta - learning_rate * gradient

    return theta


#### 3. Simulate $N=1000$ values of random variable $X_i$, distributed uniformly on interval $[-2,2]$. Simulate the values of random variable       $Y_i = 1 + 2X_i + e_i$, where $e_i$ is drawn from a Gaussian distribution $N(0, 2)$. Fit this data with linear regression, and also with ridge regression for different values of $\lambda \in \{1,10,100,1000,10000\}$. Print the slope, the MSE values, and the $R^2$ statistic for each case and write down some observations. What happens as the regularization parameter $\lambda$ increases?

In [88]:
# Simulate data
np.random.seed(42)
N = 1000
X_sim = np.random.uniform(-2, 2.0000000001, N) # effectively [-2, 2] inclusive (.uniform is [,))
e_sim = np.random.normal(1, 0.5, N)
e_sim = np.clip(e_sim, 0, 2) # force it to be (0, 2)
y_sim = 1 + 2 * X_sim + e_sim
X_sim_b = add_bias(X_sim)

# Linear regression (no regularization)
theta_linear = find_closed_form_theta(X_sim.reshape(-1, 1), y_sim)
y_pred_linear = predict(X_sim_b, theta_linear)
mse_linear = mean_squared_error(y_sim, y_pred_linear)
r2_linear = r2_score(y_sim, y_pred_linear)

print("Linear Regression:")
print(f"Slope: {theta_linear[1]:.6f}")
print(f"MSE: {mse_linear:.6f}")
print(f"R²: {r2_linear:.6f}\n")

# Ridge regression with different lambda values (with the same LR and iterations to compare the effect of regularization)
lambda_values = [1, 10, 100, 1000, 10000]

for lambda_reg in lambda_values:

    theta_ridge = gradient_descent_ridge_regression(X_sim_b, y_sim, learning_rate=0.01, iterations=100, lambda_reg=lambda_reg)
    y_pred_ridge = predict(X_sim_b, theta_ridge)
    mse_ridge = mean_squared_error(y_sim, y_pred_ridge)
    r2_ridge = r2_score(y_sim, y_pred_ridge)
    
    print(f"Ridge Regression (λ={lambda_reg}):")
    print(f"Learning Rate: {learning_rate}, Iterations: {iterations}")
    print(f"Slope: {theta_ridge[1]:.6f}")
    print(f"MSE: {mse_ridge:.6f}")
    print(f"R²: {r2_ridge:.6f}\n")

print("Observations: As λ increases, the slope decreases towards zero (started at 1.92, ended at 0.22), regularization becomes stronger, MSE on the simulated data increases + R^2 \ndecreases, and the model underfits more. The true slope is 2.0; ridge regression with high λ heavily penalizes large coefficients, shrinking the slope and increasing \nthe MSE. Ridge regression trades off accuracy on the dataset to reduce overfitting, and with high λ, the bias increases significantly, leading to underfitting.")

Linear Regression:
Slope: 1.981575
MSE: 0.224825
R²: 0.959718

Ridge Regression (λ=1):
Learning Rate: 0.5, Iterations: 100
Slope: 1.840416
MSE: 0.332005
R²: 0.940515

Ridge Regression (λ=10):
Learning Rate: 0.5, Iterations: 100
Slope: 1.830456
MSE: 0.342094
R²: 0.938707

Ridge Regression (λ=100):
Learning Rate: 0.5, Iterations: 100
Slope: 1.735612
MSE: 0.462669
R²: 0.917104

Ridge Regression (λ=1000):
Learning Rate: 0.5, Iterations: 100
Slope: 1.116930
MSE: 2.295845
R²: 0.588655

Ridge Regression (λ=10000):
Learning Rate: 0.5, Iterations: 100
Slope: 0.231693
MSE: 7.642494
R²: -0.369299

Observations: As λ increases, the slope decreases towards zero (started at 1.92, ended at 0.22), regularization becomes stronger, MSE on the simulated data increases + R^2 
decreases, and the model underfits more. The true slope is 2.0; ridge regression with high λ heavily penalizes large coefficients, shrinking the slope and increasing 
the MSE. Ridge regression trades off accuracy on the dataset to re